# Bounding Box Review

Review and filter detections per panel **before** running `extract_crops.py`.

**Workflow:**
1. Pick a panel from the dropdown  
2. Adjust the containment threshold — sub-crops (bboxes mostly inside a larger one) are auto-excluded (red)  
3. Toggle individual detections in the card grid to manually include/exclude  
4. Click **Save approved** — writes `annotated/<panel>_approved.json` (same schema, excluded detections removed)  
5. Run `extract_crops.py --approved-dir` to crop only approved detections

Approved files accumulate — you can review panels in any order.

In [1]:
# ── Cell 1: imports & paths ────────────────────────────────────────────────
import json
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image

REPO_ROOT    = Path("../..")
ANNOTATED    = REPO_ROOT / "frobenius_artifacts/analysis/annotated"
PANELS_DIR   = REPO_ROOT / "frobenius_artifacts/analysis/panels"

# Collect all panels that have both a detections JSON and a panel PNG
_jsons = sorted(ANNOTATED.glob("*_detections.json"))
_panels = []
for j in _jsons:
    stem = j.stem.replace("_detections", "")
    png  = PANELS_DIR / f"{stem}.png"
    if png.exists():
        _panels.append((stem, j, png))
    else:
        # try _cropped suffix variant
        alt = PANELS_DIR / f"{stem}_cropped.png"
        if alt.exists():
            _panels.append((stem, j, alt))

print(f"{len(_panels)} panels with both detections JSON and panel PNG")
for stem, j, png in _panels[:5]:
    dets = json.loads(j.read_text())
    print(f"  {stem}: {len(dets)} detections")

57 panels with both detections JSON and panel PNG
  EBA-B_00425_Ibadan_q97912_i1_panel_00: 3 detections
  EBA-B_00642_q98212_i3_panel_00: 8 detections
  EBA-Div_00302_q166558_i1_panel_00: 3 detections
  EBA-Div_00303_Ado_Ekiti_q166559_i1_panel_00: 1 detections
  EBA-Div_00311_Ife_q166566_i1_panel_00: 6 detections


In [ ]:
## ── Cell 2: main review UI ─────────────────────────────────────────────────
#
# Controls:
#   Panel dropdown         — pick a panel to review
#   Containment threshold  — sub-crops auto-excluded (red)
#   Min IoU                — auto-exclude low-confidence detections
#   Detection cards        — thumbnail + metadata + Include checkbox per detection
#   [Show candidates]      — toggle: show SAM masks below filter threshold (dashed yellow)
#                            requires _detections_raw.json (written by Phase 1a motif_segment.py)
#   Manual bbox section    — enter x/y/w/h coordinates and click Add to create a manual bbox
#   Save approved button   — writes annotated/<panel>_approved.json
#                            adds "source" field: sam_approved | sam_candidate | manual

# ── Containment helpers ───────────────────────────────────────────────────────
def _containment(a, b):
    """Fraction of smaller bbox covered by intersection of a and b."""
    ax1, ay1 = a["x"], a["y"]
    ax2, ay2 = ax1 + a["w"], ay1 + a["h"]
    bx1, by1 = b["x"], b["y"]
    bx2, by2 = bx1 + b["w"], by1 + b["h"]
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    if ix2 <= ix1 or iy2 <= iy1:
        return 0.0
    inter = (ix2 - ix1) * (iy2 - iy1)
    min_area = min(a["w"] * a["h"], b["w"] * b["h"])
    return inter / min_area if min_area > 0 else 0.0


def _iou_bbox(a, b):
    """IoU of two {x,y,w,h} bboxes."""
    ax1, ay1 = a["x"], a["y"]
    ax2, ay2 = ax1 + a["w"], ay1 + a["h"]
    bx1, by1 = b["x"], b["y"]
    bx2, by2 = bx1 + b["w"], by1 + b["h"]
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    if ix2 <= ix1 or iy2 <= iy1:
        return 0.0
    inter = (ix2 - ix1) * (iy2 - iy1)
    union = a["w"] * a["h"] + b["w"] * b["h"] - inter
    return inter / union if union > 0 else 0.0


def _auto_exclude(dets, threshold):
    """Return set of indices auto-excluded by containment filtering."""
    n = len(dets)
    areas = [d["bbox"]["w"] * d["bbox"]["h"] for d in dets]
    by_size = sorted(range(n), key=lambda i: -areas[i])
    suppress = set()
    for pos, i in enumerate(by_size):
        if i in suppress:
            continue
        for j in by_size[pos + 1:]:
            if j in suppress:
                continue
            if _containment(dets[i]["bbox"], dets[j]["bbox"]) >= threshold:
                suppress.add(j)
    return suppress


# ── Panel image drawing ───────────────────────────────────────────────────────
MAX_DISPLAY_W = 900

def _draw_panel(panel_img, dets, include_mask, out, candidates=None):
    """Render panel with colour-coded bboxes; optional dashed yellow candidate overlay."""
    iw, ih = panel_img.size
    scale = min(1.0, MAX_DISPLAY_W / iw)
    dw, dh = int(iw * scale), int(ih * scale)

    fig, ax = plt.subplots(figsize=(dw / 96, dh / 96), dpi=96)
    ax.imshow(panel_img.resize((dw, dh), Image.LANCZOS))
    ax.axis("off")

    for i, d in enumerate(dets):
        b = d["bbox"]
        x, y, w, h = b["x"] * scale, b["y"] * scale, b["w"] * scale, b["h"] * scale
        included = include_mask[i]
        color = "#44dd44" if included else "#ff4444"
        lw    = 2.5 if included else 1.5
        alpha = 0.9 if included else 0.55
        ax.add_patch(mpatches.Rectangle(
            (x, y), w, h,
            linewidth=lw, edgecolor=color, facecolor="none", alpha=alpha,
        ))
        ax.text(
            x + 3, y + 3, str(d["index"]),
            fontsize=max(6, int(10 * scale)), color=color, va="top", fontweight="bold",
            bbox=dict(facecolor="black", alpha=0.4, pad=1, linewidth=0),
        )

    if candidates:
        for c in candidates:
            b = c["bbox"]
            x, y, w, h = b["x"] * scale, b["y"] * scale, b["w"] * scale, b["h"] * scale
            ax.add_patch(mpatches.Rectangle(
                (x, y), w, h,
                linewidth=1.5, edgecolor="#ffcc00", facecolor="none",
                alpha=0.6, linestyle="dashed",
            ))
            ax.text(
                x + 3, y + 3, f"C{c['_cand_idx']}",
                fontsize=max(5, int(8 * scale)), color="#ffcc00", va="top",
                bbox=dict(facecolor="black", alpha=0.3, pad=1, linewidth=0),
            )

    plt.tight_layout(pad=0)
    out.clear_output(wait=True)
    with out:
        plt.show()
    plt.close(fig)


# ── State ─────────────────────────────────────────────────────────────────────
_state = {
    "stem": None,
    "dets": [],
    "candidates": [],        # Phase 1b: raw SAM masks not in filtered dets
    "panel_img": None,
    "checkboxes": [],        # one per detection
    "cand_checkboxes": [],   # one per candidate
    "auto_excl": set(),
    "dirty": False,
    "show_candidates": False,
}


# ── Widgets ───────────────────────────────────────────────────────────────────
_sl = dict(continuous_update=False,
           style={"description_width": "180px"},
           layout=widgets.Layout(width="60%"))

w_panel = widgets.Dropdown(
    options=[(s, i) for i, (s, _, __) in enumerate(_panels)],
    description="Panel:",
    style={"description_width": "60px"},
    layout=widgets.Layout(width="90%"),
)
w_contain = widgets.FloatSlider(
    min=0.0, max=1.0, step=0.05, value=0.75,
    description="containment threshold",
    readout_format=".2f", **_sl,
)
w_min_iou = widgets.FloatSlider(
    min=0.0, max=1.0, step=0.05, value=0.0,
    description="min predicted_iou",
    readout_format=".2f", **_sl,
)
btn_save = widgets.Button(
    description="Save approved",
    button_style="success",
    layout=widgets.Layout(width="160px", margin="8px 0"),
)
btn_all_in  = widgets.Button(description="Include all",   button_style="info",
                              layout=widgets.Layout(width="120px"))
btn_all_out = widgets.Button(description="Exclude all",   button_style="warning",
                              layout=widgets.Layout(width="120px"))
btn_reset   = widgets.Button(description="Reset to auto", button_style="",
                              layout=widgets.Layout(width="130px"))

# Phase 1b — candidates toggle
btn_candidates = widgets.Button(
    description="Show candidates",
    button_style="",
    layout=widgets.Layout(width="155px"),
)

# Phase 1c — manual bbox coordinate inputs
w_mx = widgets.BoundedIntText(value=0,   min=0, max=9999, description="x:",
                               layout=widgets.Layout(width="95px"))
w_my = widgets.BoundedIntText(value=0,   min=0, max=9999, description="y:",
                               layout=widgets.Layout(width="95px"))
w_mw = widgets.BoundedIntText(value=100, min=1, max=9999, description="w:",
                               layout=widgets.Layout(width="95px"))
w_mh = widgets.BoundedIntText(value=100, min=1, max=9999, description="h:",
                               layout=widgets.Layout(width="95px"))
btn_add_manual = widgets.Button(
    description="Add manual bbox",
    button_style="info",
    layout=widgets.Layout(width="145px"),
)
manual_row = widgets.HBox([
    widgets.HTML("<b style='line-height:2'>Manual&nbsp;bbox:</b>"),
    w_mx, w_my, w_mw, w_mh, btn_add_manual,
])

out_panel  = widgets.Output()
out_cards  = widgets.Output()
out_cands  = widgets.Output()
out_status = widgets.Output()


# ── Dirty state helpers (Phase 1d) ───────────────────────────────────────────
def _mark_dirty():
    _state["dirty"] = True
    btn_save.description = "Save approved ●"
    btn_save.button_style = "warning"


def _mark_clean():
    _state["dirty"] = False
    btn_save.description = "Save approved"
    btn_save.button_style = "success"


def _include_mask():
    return [cb.value for cb in _state["checkboxes"]]


def _redraw_panel():
    if _state["panel_img"] is None:
        return
    cands = _state["candidates"] if _state["show_candidates"] else None
    _draw_panel(_state["panel_img"], _state["dets"], _include_mask(), out_panel, candidates=cands)


def _on_det_change(change):
    _mark_dirty()
    _redraw_panel()


# ── Detection card builder ────────────────────────────────────────────────────
def _build_cards():
    import io
    dets      = _state["dets"]
    panel_img = _state["panel_img"]
    auto_excl = _state["auto_excl"]
    checkboxes = []
    THUMB = 96
    cards = []

    for i, d in enumerate(dets):
        b = d["bbox"]
        iw, ih = panel_img.size
        crop = panel_img.crop((max(0, b["x"]), max(0, b["y"]),
                               min(iw, b["x"] + b["w"]), min(ih, b["y"] + b["h"])))
        crop.thumbnail((THUMB, THUMB))
        sq = Image.new("RGB", (THUMB, THUMB), (30, 30, 30))
        sq.paste(crop, ((THUMB - crop.width) // 2, (THUMB - crop.height) // 2))
        buf = io.BytesIO()
        sq.save(buf, format="PNG")
        thumb_w = widgets.Image(value=buf.getvalue(), format="png", width=THUMB, height=THUMB)

        auto_tag = " ⚠ sub-crop" if i in auto_excl else ""
        src_tag  = f" [{d['source']}]" if "source" in d else ""
        cb = widgets.Checkbox(
            value=(i not in auto_excl),
            description="Include",
            indent=False,
            style={"description_width": "initial"},
            layout=widgets.Layout(width="100px"),
        )
        checkboxes.append(cb)
        cb.observe(_on_det_change, names="value")

        iou  = d.get("predicted_iou", d.get("pred_iou", "?"))
        stab = d.get("stability_score", "?")
        area = d.get("area_ratio", "?")
        meta = widgets.HTML(
            f"<div style='font-size:11px;line-height:1.5;color:#ccc'>"
            f"<b>#{d['index']}</b> {d.get('scale','?')}{src_tag}<br>"
            f"area: {area:.3f if isinstance(area, float) else area}<br>"
            f"iou: {iou:.3f if isinstance(iou, float) else iou}<br>"
            f"stab: {stab:.3f if isinstance(stab, float) else stab}<br>"
            f"{b['w']}×{b['h']} px"
            f"<span style='color:#ff8888'>{auto_tag}</span></div>"
        )
        cards.append(widgets.VBox(
            [thumb_w, meta, cb],
            layout=widgets.Layout(border="1px solid #333", padding="4px",
                                  margin="3px", width="120px", background="#1a1a1a"),
        ))

    _state["checkboxes"] = checkboxes
    COLS = 6
    rows = [widgets.HBox(cards[r:r+COLS]) for r in range(0, len(cards), COLS)]
    out_cards.clear_output(wait=True)
    with out_cards:
        display(widgets.VBox(rows) if rows else widgets.HTML("<i>No detections</i>"))


# ── Candidate card builder (Phase 1b) ────────────────────────────────────────
def _build_candidate_cards():
    import io
    cands = _state["candidates"]
    panel_img = _state["panel_img"]

    if not cands:
        out_cands.clear_output(wait=True)
        with out_cands:
            display(widgets.HTML(
                "<i style='color:#888'>No candidate pool — re-run motif_segment.py "
                "to generate _detections_raw.json</i>"
            ))
        return

    THUMB = 96
    cand_checkboxes = []
    cards = []
    for c in cands:
        b = c["bbox"]
        iw, ih = panel_img.size
        crop = panel_img.crop((max(0, b["x"]), max(0, b["y"]),
                               min(iw, b["x"] + b["w"]), min(ih, b["y"] + b["h"])))
        crop.thumbnail((THUMB, THUMB))
        sq = Image.new("RGB", (THUMB, THUMB), (20, 15, 5))
        sq.paste(crop, ((THUMB - crop.width) // 2, (THUMB - crop.height) // 2))
        buf = io.BytesIO()
        sq.save(buf, format="PNG")
        thumb_w = widgets.Image(value=buf.getvalue(), format="png", width=THUMB, height=THUMB)

        cb = widgets.Checkbox(
            value=False,
            description="Promote",
            indent=False,
            style={"description_width": "initial"},
            layout=widgets.Layout(width="100px"),
        )
        cand_checkboxes.append(cb)
        cb.observe(_on_det_change, names="value")

        iou  = c.get("predicted_iou", "?")
        area = c.get("area_ratio", "?")
        meta = widgets.HTML(
            f"<div style='font-size:11px;line-height:1.5;color:#cc9900'>"
            f"<b>C{c['_cand_idx']}</b> candidate<br>"
            f"area: {area:.3f if isinstance(area, float) else area}<br>"
            f"iou: {iou:.3f if isinstance(iou, float) else iou}<br>"
            f"{b['w']}×{b['h']} px</div>"
        )
        cards.append(widgets.VBox(
            [thumb_w, meta, cb],
            layout=widgets.Layout(border="1px solid #554400", padding="4px",
                                  margin="3px", width="120px", background="#1a0d00"),
        ))

    _state["cand_checkboxes"] = cand_checkboxes
    COLS = 6
    rows = [widgets.HBox(cards[r:r+COLS]) for r in range(0, len(cards), COLS)]
    out_cands.clear_output(wait=True)
    with out_cands:
        display(widgets.VBox([
            widgets.HTML(f"<b style='color:#ffcc00'>"
                         f"Candidate pool — {len(cands)} masks not in filtered detections</b>"),
            widgets.VBox(rows),
        ]))


def _load_candidates():
    """Load _detections_raw.json, filter out masks already in dets (by IoU)."""
    stem = _state["stem"]
    raw_path = ANNOTATED / f"{stem}_detections_raw.json"
    if not raw_path.exists():
        _state["candidates"] = []
        return

    raw = json.loads(raw_path.read_text())
    current_bboxes = [d["bbox"] for d in _state["dets"]]

    candidates = []
    cand_idx = 0
    for m in raw:
        if not any(_iou_bbox(m["bbox"], cb) > 0.3 for cb in current_bboxes):
            m["_cand_idx"] = cand_idx
            candidates.append(m)
            cand_idx += 1

    _state["candidates"] = candidates


# ── Panel loading ─────────────────────────────────────────────────────────────
def _load_panel(_=None):
    # Phase 1d: autosave on panel switch if dirty
    if _state["dirty"] and _state["stem"] is not None:
        _save(autosave=True)

    idx = w_panel.value
    stem, json_path, png_path = _panels[idx]

    # Prefer _approved.json if it exists
    approved_path = ANNOTATED / f"{stem}_approved.json"
    if approved_path.exists():
        dets = json.loads(approved_path.read_text())
        _src = "approved"
    else:
        dets = json.loads(json_path.read_text())
        _src = "detections"

    panel_img = Image.open(png_path).convert("RGB")

    min_iou = w_min_iou.value
    dets = [d for d in dets
            if d.get("predicted_iou", d.get("pred_iou", 1.0)) >= min_iou]

    auto_excl = _auto_exclude(dets, w_contain.value)

    _state.update(
        stem=stem, dets=dets, panel_img=panel_img,
        auto_excl=auto_excl, candidates=[], cand_checkboxes=[],
    )
    _mark_clean()

    _build_cards()
    _redraw_panel()
    out_cands.clear_output()
    out_status.clear_output()
    with out_status:
        print(f"{stem} — {len(dets)} detections, "
              f"{len(auto_excl)} auto-excluded  [from {_src}]")

    if _state["show_candidates"]:
        _load_candidates()
        _build_candidate_cards()
        _redraw_panel()


def _on_threshold(_=None):
    dets = _state["dets"]
    if not dets:
        return
    prev = _include_mask()
    _state["auto_excl"] = _auto_exclude(dets, w_contain.value)
    _build_cards()
    for i, cb in enumerate(_state["checkboxes"]):
        cb.value = prev[i] if i < len(prev) else (i not in _state["auto_excl"])
    _redraw_panel()


def _include_all(_):
    for cb in _state["checkboxes"]: cb.value = True
    _mark_dirty()


def _exclude_all(_):
    for cb in _state["checkboxes"]: cb.value = False
    _mark_dirty()


def _reset_to_auto(_):
    auto = _state["auto_excl"]
    for i, cb in enumerate(_state["checkboxes"]): cb.value = (i not in auto)
    _mark_dirty()


def _on_candidates_toggle(_):
    """Phase 1b: toggle candidate pool display."""
    _state["show_candidates"] = not _state["show_candidates"]
    if _state["show_candidates"]:
        btn_candidates.description = "Hide candidates"
        btn_candidates.button_style = "warning"
        _load_candidates()
        _build_candidate_cards()
    else:
        btn_candidates.description = "Show candidates"
        btn_candidates.button_style = ""
        out_cands.clear_output()
    _redraw_panel()


def _on_add_manual(_):
    """Phase 1c: add a manual bbox from coordinate inputs."""
    x, y, w, h = w_mx.value, w_my.value, w_mw.value, w_mh.value
    if w <= 0 or h <= 0:
        out_status.clear_output()
        with out_status: print("  ⚠ width and height must be > 0")
        return

    panel_img = _state["panel_img"]
    if panel_img is None:
        return

    pw, ph = panel_img.size
    area_ratio = (w * h) / (pw * ph) if pw * ph > 0 else 0.0

    prev_mask = _include_mask()
    new_idx = len(_state["dets"])
    _state["dets"].append({
        "index": new_idx,
        "bbox": {"x": x, "y": y, "w": w, "h": h},
        "scale": "motif" if area_ratio < 0.25 else "register",
        "area_ratio": round(area_ratio, 5),
        "predicted_iou": 1.0,
        "stability_score": 1.0,
        "source": "manual",
    })
    _build_cards()
    for i, cb in enumerate(_state["checkboxes"]):
        if i < len(prev_mask):
            cb.value = prev_mask[i]
        # new det defaults to included (True)

    _redraw_panel()
    _mark_dirty()
    out_status.clear_output()
    with out_status:
        print(f"  Added manual bbox #{new_idx}: ({x},{y}) {w}×{h}")


def _save(btn_event=None, autosave=False):
    dets  = _state["dets"]
    stem  = _state["stem"]
    if not stem:
        return
    mask  = _include_mask()

    # Collect promoted candidates (Phase 1b)
    promoted = []
    for ci, cb in enumerate(_state.get("cand_checkboxes", [])):
        if cb.value and ci < len(_state["candidates"]):
            cand = dict(_state["candidates"][ci])
            cand.pop("_cand_idx", None)
            cand["source"] = "sam_candidate"
            cand["index"] = sum(1 for inc in mask if inc) + len(promoted)
            promoted.append(cand)

    approved = []
    for d, inc in zip(dets, mask):
        if inc:
            d = dict(d)
            if "source" not in d:
                d["source"] = "sam_approved"
            approved.append(d)
    approved.extend(promoted)

    out_path = ANNOTATED / f"{stem}_approved.json"
    out_path.write_text(json.dumps(approved, indent=2))
    _mark_clean()

    out_status.clear_output()
    with out_status:
        tag = "Auto-saved" if autosave else "Saved"
        print(f"{tag}: {len(approved)}/{len(dets)} approved → {out_path.name}")
        if promoted:
            print(f"  + {len(promoted)} promoted candidate(s)")


w_panel.observe(_load_panel, names="value")
w_contain.observe(_on_threshold, names="value")
w_min_iou.observe(_load_panel, names="value")
btn_save.on_click(_save)
btn_all_in.on_click(_include_all)
btn_all_out.on_click(_exclude_all)
btn_reset.on_click(_reset_to_auto)
btn_candidates.on_click(_on_candidates_toggle)
btn_add_manual.on_click(_on_add_manual)

display(
    widgets.HTML("<h3 style='margin:4px 0'>Bounding Box Review</h3>"),
    w_panel,
    widgets.HBox([w_contain, w_min_iou]),
    widgets.HBox([btn_all_in, btn_all_out, btn_reset, btn_save, btn_candidates]),
    manual_row,
    out_status,
    out_panel,
    widgets.HTML("<b style='font-size:13px'>Detection cards — toggle to include/exclude</b>"),
    out_cards,
    widgets.HTML("<b style='font-size:13px;color:#ffcc00'>"
                 "Candidate pool (SAM masks below filter threshold)</b>"),
    out_cands,
)
_load_panel()